# C — Khoai tây: PlantVillage → PlantDoc, test giữ riêng 50% (dữ liệu của Mahapatra et al.)

Đặt cạnh **Mahapatra et al. 2026**, *Advancing plant disease classification using an attention-based CNN for intra-dataset and
cross-dataset training*, Scientific Reports 16:10925 (Table 21).

| | Mahapatra et al. | Notebook này |
|---|---|---|
| Source | PlantVillage khoai tây 2,152 ảnh (có cả healthy); cross-dataset dùng early + late blight | early + late blight, **600** ảnh (300 / lớp), qua FBR |
| Target | PlantDoc khoai tây, **222** ảnh | PlantDoc gộp train + test, bỏ ảnh trùng MD5 → **212** ảnh |
| Test | "50% của PlantDoc" (không rõ là cả 222 ảnh hay 50%) | **50% stratified** = 106 ảnh (56 early / 50 late); 50% còn lại làm target **không nhãn** |
| Mô hình | attention-CNN, ảnh 150 px, Adam, batch 32, 100 epoch, **không thích nghi** | ResNet-18, 224 px, FBR + CDAN+E + MT, batch 32, 300 epoch |
| Số liệu báo | **chỉ theo lớp**: early 94.87, late 10.48 | accuracy, theo lớp, trung bình lớp |

### Kết quả đã chạy (test 106 ảnh, seed 42, SAM ViT-H, ResNet-18)

| Nhánh | early blight | late blight | Trung bình 2 lớp | Accuracy |
|---|---|---|---|---|
| CDAN+E + MT | 64.29 | 70.00 | 67.14 | **66.98** |
| *Mahapatra, không thích nghi* | *94.87* | *10.48* | *52.68* * | – |

\* trung bình tự tính; bài chỉ báo cáo theo lớp. Mô hình của bài gần như đoán mọi ảnh là early blight (94.87 / 10.48 ≈ bộ phân loại hằng),
nên accuracy theo lớp của họ không phản ánh khả năng nhận diện; trung bình recall các lớp là con số công bằng cho bài toán 2 lớp.
Test chỉ 106 ảnh: một ảnh ≈ 0.9 điểm, khoảng tin cậy 95% khoảng ±9 điểm. **Mới chạy nhánh CDAN+E + MT**; các nhánh còn lại (`src_raw`, `src_fbr`,
`cdane`, `mt_only`) chưa có, nên chưa tách được đóng góp của từng thành phần. SAM: file đã tải cho thấy cell **ViT-H** được chạy (cell ViT-B không chạy).

### Chạy lại để ra số này

1. §2: `SAM_MODEL_TYPE = 'vit_h'` (cache FBR cũ `fbr_cache_d42_n300` được dùng lại).
2. Chạy từ đầu đến hết §7 (kiểm tra ghép cặp RNG), rồi §8 với `RUN_ARMS = ['cdane_mt']`. Muốn đủ bảng ablation: `RUN_ARMS = list(ARMS)`.
3. Kết quả (JSON) và checkpoint ghi vào `results_<sam>/`, `checkpoints_<sam>/`; §9 so sánh các nhánh đã có.

### Đồng bộ giữa ba notebook A, B, C

- Cùng cấu trúc: **§2 cấu hình → §3 dữ liệu → §4 SAM + FBR → loaders → models → huấn luyện → kết quả**.
- Cùng **công tắc `SAM_MODEL_TYPE`** (`vit_b` | `vit_h`, đặt ở §2): nó chọn cache FBR và thư mục kết quả / checkpoint theo SAM,
  nên chạy lại với SAM khác không ghi đè hay bỏ qua nhầm kết quả của SAM kia.
- Cùng thiết lập huấn luyện: AdamW lr 1e-3, weight decay 0.01, cosine, 300 epoch, chọn model theo val loss trên source từ epoch 250
  (không dùng nhãn target), seed 42; Mean Teacher: EMA 0.99, `w(t)` bằng 0 đến epoch 100 rồi tăng tới `C²` trong 50 epoch.
- **Các số ở bảng trên là output đã lưu trong chính notebook này.** Chạy lại cùng cấu hình (cùng SAM, backbone, seed 42) cho kết quả
  tương ứng; khác phần cứng / phiên bản PyTorch có thể lệch nhỏ vì một số phép toán GPU không hoàn toàn tất định.

## 1 — Setup & Reproducibility

In [1]:
!pip install segment-anything -q        # SAM cho FBR

import os, random, copy, json, hashlib, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as tv_models
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')


def set_seed(seed: int = 42):
    # Seed moi nguon ngau nhien (Python, NumPy, PyTorch CPU/GPU) + cuDNN tat dinh.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'


def seed_worker(worker_id: int):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


# Generator rieng cho shuffle cua DataLoader. set_seed() KHONG dung toi g,
# nen vong lap huan luyen re-seed g moi epoch (xem train_arm).
g = torch.Generator()
g.manual_seed(42)
print('Setup complete.')

Device : cuda
GPU    : Tesla T4
Setup complete.


## 2 — Drive, đường dẫn & cấu hình

`TASK` chọn cặp dữ liệu. `potato` là thực nghiệm chính (có số liệu Table 21 của bài 5
để đặt cạnh). `corn` để sẵn cho PV ngô → PlantDoc ngô (3 lớp chung), **bài 5 không có
số liệu train trên PV cho ngô**, nên nhánh đó chỉ dùng để so các nhánh với nhau.

PlantVillage được clone **một lần** ở thư mục gốc (dùng chung với notebook táo); mọi
thứ riêng của task (PlantDoc, FBR cache, checkpoint, kết quả) nằm trong `TASK_DIR`.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# ── SAM cho FBR: 'vit_b' | 'vit_h'. Quyet dinh cache FBR va thu muc ket qua / checkpoint ──
SAM_MODEL_TYPE = 'vit_h'

SAVE_DIR = '/content/drive/MyDrive/GRAD PACKAGE/THESIS/PAPER RESEARCH/CDAN+E combine MT (Corn DS)'

# ── Chọn task ────────────────────────────────────────────────────────────────
TASK = 'potato'

TASKS = {
    'potato': {
        'class_names': ['early_blight', 'late_blight'],
        # PlantVillage (raw/color/<folder>) -> label
        'pvd_folders': {'Potato___Early_blight': 0, 'Potato___Late_blight': 1},
        # PlantDoc (train|test/<folder>) -> label
        'plantdoc_folders': {'Potato leaf early blight': 0, 'Potato leaf late blight': 1},
        # Bai 5, Table 21: train PlantVillage, test PlantDoc, accuracy theo lop (%)
        'paper_ref': {'early_blight': 94.87, 'late_blight': 10.48},
    },
    'corn': {
        'class_names': ['gray_leaf_spot', 'common_rust', 'northern_leaf_blight'],
        'pvd_folders': {
            'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 0,
            'Corn_(maize)___Common_rust_': 1,
            'Corn_(maize)___Northern_Leaf_Blight': 2,
        },
        'plantdoc_folders': {'Corn Gray leaf spot': 0, 'Corn rust leaf': 1, 'Corn leaf blight': 2},
        'paper_ref': None,   # bai 5 khong train tren PV cho ngo
    },
}
CFG         = TASKS[TASK]
CLASS_NAMES = CFG['class_names']
NUM_CLASSES = len(CLASS_NAMES)

# ── Du lieu ──────────────────────────────────────────────────────────────────
DATA_SEED      = 42     # co dinh: split source, split PlantDoc, FBR cache
MAX_PER_CLASS  = 300    # anh PV moi lop (giong notebook tao)
SRC_VAL_RATIO  = 0.25   # PV train/val 75/25 (giong notebook tao)
TEST_RATIO     = 0.50   # PlantDoc test 50% (giong bai 5)
BATCH_SIZE     = 32     # bai 5 dung 32; target ~111 anh nen 64 qua lon
PLANTDOC_MAX_SIDE = 640 # resize truoc anh PlantDoc (anh goc co the > 3000px)

# ── Huan luyen ───────────────────────────────────────────────────────────────
SEEDS      = [42]       # nen chay 3-5 seed: test chi ~111 anh, 1 anh ~ 0.9 diem
MAX_EPOCHS = 300
MIN_SEL    = 250        # chi chon model tu epoch nay (giong notebook tao)
LR, WD     = 1e-3, 0.01
LOG_EVERY  = 10         # cung la chu ky luu checkpoint

# ── Duong dan ────────────────────────────────────────────────────────────────
PLANT_VILLAGE_SAVE_PATH = os.path.join(SAVE_DIR, 'PlantVillage')
PLANT_VILLAGE_DIR       = os.path.join(PLANT_VILLAGE_SAVE_PATH, 'raw', 'color')
PLANTDOC_SAVE_PATH      = os.path.join(SAVE_DIR, 'PlantDoc')

TASK_DIR         = os.path.join(SAVE_DIR, f'plantdoc_{TASK}')
PLANTDOC_RESIZED = os.path.join(TASK_DIR, 'plantdoc_resized')
# ViT-H dung lai cache cu 'fbr_cache_d..' (da tinh bang ViT-H); ViT-B dung thu muc rieng.
FBR_CACHE_DIR    = os.path.join(TASK_DIR, f'fbr_cache_d{DATA_SEED}_n{MAX_PER_CLASS}' if SAM_MODEL_TYPE == 'vit_h' else f'fbr_b_cache_d{DATA_SEED}_n{MAX_PER_CLASS}')
CKPT_DIR         = os.path.join(TASK_DIR, f'checkpoints_{SAM_MODEL_TYPE}')
RESULTS_DIR      = os.path.join(TASK_DIR, f'results_{SAM_MODEL_TYPE}')
for _d in (SAVE_DIR, TASK_DIR, PLANTDOC_RESIZED, FBR_CACHE_DIR, CKPT_DIR, RESULTS_DIR):
    os.makedirs(_d, exist_ok=True)

print(f'TASK={TASK}  classes={CLASS_NAMES}')
print(f'TASK_DIR = {TASK_DIR}')

Mounted at /content/drive
TASK=potato  classes=['early_blight', 'late_blight']
TASK_DIR = /content/drive/MyDrive/GRAD PACKAGE/THESIS/PAPER RESEARCH/CDAN+E combine MT (Corn DS)/plantdoc_potato


## 3 — Tải dữ liệu

Cả hai đều trên GitHub, **không cần Kaggle token**.

In [3]:
def _clone_if_missing(url, dest):
    if os.path.isdir(dest) and os.listdir(dest):
        print(f'{os.path.basename(dest)} da co, bo qua.')
        return
    os.makedirs(dest, exist_ok=True)
    print(f'Cloning {url} ...')
    !git clone "{url}" "{dest}" --depth=1 -q
    print('Xong.')

# _clone_if_missing('https://github.com/spMohanty/PlantVillage-Dataset.git', PLANT_VILLAGE_SAVE_PATH)
_clone_if_missing('https://github.com/pratikkayal/PlantDoc-Dataset.git', PLANTDOC_SAVE_PATH)

Cloning https://github.com/pratikkayal/PlantDoc-Dataset.git ...
Xong.


### 3.1 — Transforms & Dataset class (giống hệt notebook táo)

In [4]:
_NORM = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

TRAIN_TRANSFORM = T.Compose([
    T.Resize((256, 256)),
    T.RandomCrop(224),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    _NORM,
])

VAL_TRANSFORM = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    _NORM,
])


class PlantDiseaseDataset(Dataset):
    # samples: list (path, label). labeled=False -> tra ve -1 thay cho nhan,
    # chan ro ri nhan target vao huan luyen.
    def __init__(self, samples, classes, transform=None, labeled=True):
        self.samples      = samples
        self.classes      = classes
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.transform    = transform
        self.labeled      = labeled

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return (image, label) if self.labeled else (image, -1)

    def get_labels(self):
        return [s[1] for s in self.samples]

print('Transforms and dataset class defined.')

Transforms and dataset class defined.


### 3.2 — Load & split

| Tập | Nguồn | Kích thước (khoai tây) | Dùng nhãn? | Vai trò |
|---|---|---|---|---|
| source train | PV, 75% | ~450 | có | `L_cls` |
| source val | PV, 25% | ~150 | có | chọn model (val loss) |
| target | PlantDoc, 50% | ~111 | **không** | `L_cdan`, `L_cons`, **pool ảnh nền FBR** |
| test | PlantDoc, 50% | ~111 | chỉ để đánh giá cuối | khớp phần test 50% của bài 5 |

**Vì sao gộp `train/` + `test/` của PlantDoc rồi chia lại:** split có sẵn của repo
PlantDoc không phải 50/50; bài 5 tự chia 35/15/50. Chia stratified theo `DATA_SEED`
giữ tỉ lệ hai lớp như nhau ở target và test.

**Một điểm mơ hồ của bài 5:** bài viết "tested on the images of PlantDoc" — không rõ là
toàn bộ 222 ảnh hay chỉ phần test 50%. Notebook dùng phần 50% (an toàn: không rò
test vào huấn luyện). Hãy nêu rõ lựa chọn này trong luận văn.

In [5]:
VALID_EXTS = ('.jpg', '.jpeg', '.png')


def _find_folder(root, name):
    # Ten thu muc lech nhau giua cac ban phat hanh -> fallback khong phan biet hoa thuong.
    exact = os.path.join(root, name)
    if os.path.isdir(exact):
        return exact
    for f in os.listdir(root):
        if f.lower() == name.lower() or name.lower() in f.lower():
            return os.path.join(root, f)
    return None


def load_pvd_samples(data_dir, class_map, max_per_class, seed):
    # Lay toi da max_per_class anh moi lop, xao tron bang RNG rieng (khong dung RNG toan cuc).
    rng = random.Random(seed)
    samples = []
    print('[PVD] Source (lab):')
    for folder_name, label in class_map.items():
        folder = _find_folder(data_dir, folder_name)
        if folder is None:
            raise FileNotFoundError(f'Khong thay thu muc PV: {folder_name} trong {data_dir}')
        imgs = sorted(os.path.join(folder, f) for f in os.listdir(folder)
                      if f.lower().endswith(VALID_EXTS))
        rng.shuffle(imgs)
        selected = imgs[:max_per_class]
        samples += [(p, label) for p in selected]
        print(f'  {folder_name:<52}: {len(imgs):>5} co -> {len(selected):>4} dung')
    print(f'  Tong PVD: {len(samples)}')
    return samples


def _resize_copy(src, dst, max_side):
    # Anh PlantDoc co the rat lon; resize mot lan de huan luyen khong phai decode anh 4000px.
    # Tra ve False neu anh hong (bo qua anh do).
    if os.path.exists(dst):
        return True
    try:
        img = Image.open(src).convert('RGB')
    except Exception as e:
        print(f'  [skip] khong doc duoc {os.path.basename(src)}: {e}')
        return False
    img.thumbnail((max_side, max_side))
    img.save(dst, quality=95)
    return True


def load_plantdoc_samples(root, class_map, out_dir, max_side):
    # Gop train/ + test/ cua PlantDoc, loai anh trung MD5, resize vao out_dir.
    by_hash = {}            # md5 -> (src_path, label)
    conflicts = set()       # md5 xuat hien voi >1 nhan -> bo
    n_files = 0
    for split in ('train', 'test'):
        split_dir = os.path.join(root, split)
        for folder_name, label in class_map.items():
            folder = _find_folder(split_dir, folder_name)
            if folder is None:
                print(f'  WARNING: khong thay {split}/{folder_name}')
                continue
            for f in sorted(os.listdir(folder)):
                if not f.lower().endswith(VALID_EXTS):
                    continue
                path = os.path.join(folder, f)
                n_files += 1
                with open(path, 'rb') as fh:
                    h = hashlib.md5(fh.read()).hexdigest()
                if h in by_hash and by_hash[h][1] != label:
                    conflicts.add(h)
                by_hash.setdefault(h, (path, label))

    samples = []
    for h, (path, label) in sorted(by_hash.items(), key=lambda kv: kv[1][0]):
        if h in conflicts:
            continue
        dst = os.path.join(out_dir, f'{h}.jpg')
        if _resize_copy(path, dst, max_side):
            samples.append((dst, label))

    print(f'[PlantDoc] Target (field): {n_files} file -> {len(by_hash)} anh duy nhat '
          f'-> bo {len(conflicts)} anh trung khac nhan -> {len(samples)} dung')
    return samples


def split_target(samples, test_ratio, seed):
    # Chia stratified PlantDoc thanh (target khong nhan, test). Test KHONG BAO GIO dung de train/chon model.
    idx = list(range(len(samples)))
    tgt_idx, test_idx = train_test_split(
        idx, test_size=test_ratio, stratify=[s[1] for s in samples], random_state=seed
    )
    return [samples[i] for i in tgt_idx], [samples[i] for i in test_idx]


def make_loader(dataset, batch_size, shuffle=False, drop_last=False):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      pin_memory=torch.cuda.is_available(), drop_last=drop_last,
                      worker_init_fn=seed_worker, generator=g)


def _count(samples):
    c = np.bincount([s[1] for s in samples], minlength=NUM_CLASSES)
    return ', '.join(f'{n}={k}' for n, k in zip(CLASS_NAMES, c))

print('Load/split functions defined.')

Load/split functions defined.


In [13]:
pvd_samples      = load_pvd_samples(PLANT_VILLAGE_DIR, CFG['pvd_folders'], MAX_PER_CLASS, DATA_SEED)
plantdoc_samples = load_plantdoc_samples(PLANTDOC_SAVE_PATH, CFG['plantdoc_folders'],
                                         PLANTDOC_RESIZED, PLANTDOC_MAX_SIDE)

target_samples, test_samples = split_target(plantdoc_samples, TEST_RATIO, DATA_SEED)

# Source train/val: cung chi so cho ban PV goc va ban FBR (FBR cache 1-1 voi pvd_samples).
src_tr_idx, src_vl_idx = train_test_split(
    list(range(len(pvd_samples))), test_size=SRC_VAL_RATIO,
    stratify=[s[1] for s in pvd_samples], random_state=DATA_SEED,
)

assert not set(p for p, _ in target_samples) & set(p for p, _ in test_samples), 'target/test trung anh'

print(f'\n{"=" * 64}\nDATA SUMMARY ({TASK})\n{"=" * 64}')
print(f'  PV  train : {len(src_tr_idx):>4}  ({_count([pvd_samples[i] for i in src_tr_idx])})')
print(f'  PV  val   : {len(src_vl_idx):>4}  ({_count([pvd_samples[i] for i in src_vl_idx])})')
print(f'  PD  target: {len(target_samples):>4}  ({_count(target_samples)})  KHONG dung nhan')
print(f'  PD  test  : {len(test_samples):>4}  ({_count(test_samples)})')
print('=' * 64)

[PVD] Source (lab):
  Potato___Early_blight                               :  1000 co ->  300 dung
  Potato___Late_blight                                :  1000 co ->  300 dung
  Tong PVD: 600
[PlantDoc] Target (field): 222 file -> 217 anh duy nhat -> bo 5 anh trung khac nhan -> 212 dung

DATA SUMMARY (potato)
  PV  train :  450  (early_blight=225, late_blight=225)
  PV  val   :  150  (early_blight=75, late_blight=75)
  PD  target:  106  (early_blight=56, late_blight=50)  KHONG dung nhan
  PD  test  :  106  (early_blight=56, late_blight=50)


## 4 — FBR: Field-adaptive Background Recomposition

Giống notebook táo: SAM (prompt điểm giữa) → `refine_mask` → ghép lá lên một mảnh
ảnh nền field. **Pool ảnh nền = target split của PlantDoc**, không bao giờ là test.

Khác một chỗ: ảnh nền đọc bằng PIL thay vì `cv2.imread`, vì PlantDoc có vài file
đuôi `.jpg` nhưng định dạng khác, `cv2.imread` trả `None` mà không báo lỗi.

In [14]:
import cv2

# ── Download and load SAM (SAM_MODEL_TYPE dat o muc 2) ────────────────────────
SAM_FILES = {
    'vit_b': ('sam_vit_b_01ec64.pth', 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'),
    'vit_h': ('sam_vit_h_4b8939.pth', 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth'),
}
_sam_file, _sam_url = SAM_FILES[SAM_MODEL_TYPE]
SAM_CHECKPOINT = f'/content/{_sam_file}'

if not os.path.exists(SAM_CHECKPOINT):
    print(f'Downloading SAM checkpoint ({SAM_MODEL_TYPE})...')
    !wget -q {_sam_url} -O {SAM_CHECKPOINT}.part && mv {SAM_CHECKPOINT}.part {SAM_CHECKPOINT}
    print('SAM checkpoint downloaded.')

from segment_anything import sam_model_registry, SamPredictor
sam      = sam_model_registry[SAM_MODEL_TYPE](checkpoint=SAM_CHECKPOINT).to(DEVICE)
sam_pred = SamPredictor(sam)
print(f'SAM {SAM_MODEL_TYPE} loaded on {DEVICE}.')

SAM checkpoint downloaded!
SAM loaded on cuda


In [15]:
def _read_rgb(path):
    return np.array(Image.open(path).convert('RGB'))


def segment_leaf(predictor, image_rgb):
    # SAM voi 1 diem prompt o tam anh (la PV luon nam giua).
    h, w = image_rgb.shape[:2]
    predictor.set_image(image_rgb)
    masks, scores, _ = predictor.predict(
        point_coords=np.array([[w // 2, h // 2]]),
        point_labels=np.array([1]),
        multimask_output=True,
    )
    return masks[np.argmax(scores)]


def refine_mask(mask, k_d=15, k_e=10, k_g=5, iterations=2):
    # Dilate -> erode (x2) roi Gaussian blur: lap lo, lam muot bien, alpha mem.
    m  = mask.astype(np.uint8) * 255
    kd = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_d, k_d))
    ke = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_e, k_e))
    for _ in range(iterations):
        m = cv2.dilate(m, kd)
        m = cv2.erode(m, ke)
    m = cv2.GaussianBlur(m, (k_g, k_g), 0)
    return m.astype(np.float32) / 255.0


def composite(lab_img, mask, bg_img):
    h, w  = lab_img.shape[:2]
    bg    = cv2.resize(bg_img, (w, h))
    alpha = mask[:, :, np.newaxis]
    out   = alpha * lab_img.astype(np.float32) + (1 - alpha) * bg.astype(np.float32)
    return np.clip(out, 0, 255).astype(np.uint8)


def random_field_background(bg_path, out_w, out_h):
    # Crop ngau nhien 30-70% canh ngan -> xoay ngau nhien -> center-crop 1/sqrt(2) (bo goc den).
    img  = _read_rgb(bg_path)
    H, W = img.shape[:2]
    s = int(random.uniform(0.30, 0.70) * min(H, W))
    x = random.randint(0, W - s)
    y = random.randint(0, H - s)
    patch = img[y:y + s, x:x + s]
    M = cv2.getRotationMatrix2D((s / 2, s / 2), random.uniform(0, 360), 1.0)
    patch = cv2.warpAffine(patch, M, (s, s))
    c = int(s / np.sqrt(2))
    o = (s - c) // 2
    patch = patch[o:o + c, o:o + c]
    return cv2.resize(patch, (out_w, out_h))


def fbr_single(lab_path, bg_paths, predictor):
    lab_rgb = _read_rgb(lab_path)
    h, w    = lab_rgb.shape[:2]
    mask    = refine_mask(segment_leaf(predictor, lab_rgb))
    bg_rgb  = random_field_background(random.choice(bg_paths), w, h)
    return composite(lab_rgb, mask, bg_rgb)

print('FBR functions defined.')

FBR functions defined.


In [16]:
# ── Pre-compute FBR cache (idempotent: chay lai chi tinh anh con thieu) ─────
fbr_bg_paths = [p for p, _ in target_samples]          # CHI target split
print(f'Pool anh nen (PlantDoc target): {len(fbr_bg_paths)} anh')
print(f'FBR cho {len(pvd_samples)} anh source -> {FBR_CACHE_DIR}')

set_seed(DATA_SEED)   # chon nen + goc xoay tai lap duoc
fbr_samples = []
n_new = n_skip = n_fail = 0
for idx, (orig_path, label) in enumerate(tqdm(pvd_samples)):
    cache_path = os.path.join(FBR_CACHE_DIR, f'{idx:05d}_c{label}.jpg')
    if os.path.exists(cache_path):
        n_skip += 1
    else:
        try:
            Image.fromarray(fbr_single(orig_path, fbr_bg_paths, sam_pred)).save(cache_path, quality=92)
        except Exception as e:
            # Khong am tham: dem lai de biet bao nhieu anh roi ve ban goc.
            print(f'  [fallback] {os.path.basename(orig_path)}: {e}')
            Image.open(orig_path).convert('RGB').save(cache_path, quality=92)
            n_fail += 1
        n_new += 1
    fbr_samples.append((cache_path, label))

print(f'FBR xong. Moi: {n_new} (fallback anh goc: {n_fail}) | Dung lai: {n_skip}')

Pool anh nen (PlantDoc target): 106 anh
FBR cho 600 anh source -> /content/drive/MyDrive/GRAD PACKAGE/THESIS/PAPER RESEARCH/CDAN+E combine MT (Corn DS)/plantdoc_potato/fbr_cache_d42_n300


  0%|          | 0/600 [00:00<?, ?it/s]

FBR xong. Moi: 600 (fallback anh goc: 0) | Dung lai: 0


In [17]:
# ── Kiem tra bang mat: anh FBR co giong anh field khong? ──
_rng = random.Random(0)
n = 6
idxs = _rng.sample(range(len(fbr_samples)), n)
fig, axes = plt.subplots(2, n, figsize=(3 * n, 6.5))
for j, idx in enumerate(idxs):
    axes[0, j].imshow(Image.open(pvd_samples[idx][0]).convert('RGB'))
    axes[0, j].set_title(f'PV {CLASS_NAMES[pvd_samples[idx][1]]}', fontsize=9); axes[0, j].axis('off')
    axes[1, j].imshow(Image.open(fbr_samples[idx][0]).convert('RGB'))
    axes[1, j].set_title('FBR', fontsize=9); axes[1, j].axis('off')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for j, (p, l) in enumerate(_rng.sample(target_samples, 4)):
    axes[j].imshow(Image.open(p).convert('RGB'))
    axes[j].set_title(f'PlantDoc target ({CLASS_NAMES[l]})', fontsize=9); axes[j].axis('off')
plt.tight_layout(); plt.show()

Output hidden; open in https://colab.research.google.com to view.

### 4.1 — Loaders

- `SOURCE['fbr']` / `SOURCE['raw']`: cùng chỉ số train/val, chỉ khác ảnh (FBR hay gốc).
- `target_loader_3v`: `TargetThreeViewDataset` của notebook táo — trả `(plain, view1, view2)`;
  `plain` cho CDAN+E, hai view sinh trong `fork_rng` cho Mean Teacher. **Mọi** nhánh đều
  đọc loader này (kể cả nhánh không dùng target) để tiêu thụ RNG giống hệt nhau.

In [18]:
class TargetThreeViewDataset(Dataset):
    # Target khong nhan -> ((plain, view1, view2), -1).
    # plain: VAL_TRANSFORM, tat dinh. view1/view2: TRAIN_TRANSFORM sinh trong fork_rng,
    # seed theo (base_seed, epoch, idx) -> khong dung RNG toan cuc, tai lap duoc sau resume.
    def __init__(self, samples, plain_transform, aug_transform, base_seed):
        self.samples         = samples
        self.plain_transform = plain_transform
        self.aug_transform   = aug_transform
        self.base_seed       = base_seed
        self.epoch           = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, _ = self.samples[idx]
        img   = Image.open(path).convert('RGB')
        plain = self.plain_transform(img)
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed((self.base_seed * 1_000_003 + self.epoch * 10_007 + idx) % (2**31 - 1))
            v1 = self.aug_transform(img)
            v2 = self.aug_transform(img)
        return (plain, v1, v2), -1


def _source_loaders(samples):
    ds_tr = PlantDiseaseDataset(samples, CLASS_NAMES, TRAIN_TRANSFORM)
    ds_vl = PlantDiseaseDataset(samples, CLASS_NAMES, VAL_TRANSFORM)
    return (make_loader(Subset(ds_tr, src_tr_idx), BATCH_SIZE, shuffle=True, drop_last=True),
            make_loader(Subset(ds_vl, src_vl_idx), BATCH_SIZE, shuffle=False))

SOURCE = {'fbr': _source_loaders(fbr_samples), 'raw': _source_loaders(pvd_samples)}

dataset_tgt_3v   = TargetThreeViewDataset(target_samples, VAL_TRANSFORM, TRAIN_TRANSFORM, DATA_SEED)
target_loader_3v = make_loader(dataset_tgt_3v, BATCH_SIZE, shuffle=True, drop_last=True)
test_loader      = make_loader(PlantDiseaseDataset(test_samples, CLASS_NAMES, VAL_TRANSFORM),
                               BATCH_SIZE, shuffle=False)

assert len(target_loader_3v) >= 1, (
    f'Target {len(target_samples)} anh < BATCH_SIZE {BATCH_SIZE} -> giam BATCH_SIZE')

# ── Kiem tra tai cho (giong notebook tao) ───────────────────────────────────
(_p, _v1, _v2), _l = next(iter(target_loader_3v))
assert _p.shape == _v1.shape == _v2.shape == (BATCH_SIZE, 3, 224, 224)
assert not torch.allclose(_v1, _v2), 'view1 == view2 -> L_cons luon = 0'
assert not torch.allclose(_p, _v1), 'plain dang bi augment'
assert (_l == -1).all(), 'target loader tra ve nhan that'

torch.manual_seed(123); _before = torch.rand(3)
torch.manual_seed(123); dataset_tgt_3v.set_epoch(7); _ = dataset_tgt_3v[0]; _after = torch.rand(3)
assert torch.equal(_before, _after), 'fork_rng ro ri sang RNG toan cuc'
del _p, _v1, _v2, _l, _before, _after

print(f'source train: {len(SOURCE["fbr"][0])} batch/epoch | '
      f'target: {len(target_loader_3v)} batch/luot (lap lai khi het) | '
      f'test: {len(test_samples)} anh')

source train: 14 batch/epoch | target: 3 batch/luot (lap lai khi het) | test: 106 anh


## 5 — Models, CDAN+E & Mean Teacher helpers (giống notebook táo)

In [19]:
def get_lambda(step, max_steps, gamma=10.0):
    p = step / max_steps
    return 2.0 / (1.0 + np.exp(-gamma * p)) - 1.0


def domain_lbl(n, is_src, device):
    return torch.full((n,), 0 if is_src else 1, dtype=torch.long, device=device)


class GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.clone()

    @staticmethod
    def backward(ctx, grad):
        return -ctx.lam * grad, None


class GradRevLayer(nn.Module):
    def __init__(self, lam=1.0):
        super().__init__()
        self.lam = lam
    def forward(self, x):
        return GRL.apply(x, self.lam)
    def set_lam(self, lam):
        self.lam = lam


class FeatureExtractor(nn.Module):
    # ResNet-18 ImageNet, bo FC -> 512-d.
    def __init__(self, pretrained=True):
        super().__init__()
        weights = tv_models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        net = tv_models.resnet18(weights=weights)
        self.out_dim = 512
        self.net = nn.Sequential(*list(net.children())[:-1])
    def forward(self, x):
        return self.net(x).flatten(1)


class Classifier(nn.Module):
    def __init__(self, in_dim=512, num_classes=NUM_CLASSES):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes)
    def forward(self, x):
        return self.fc(x)


class CDANDisc(nn.Module):
    # Input = multilinear map, dim = 512 * num_classes. Hai lop an 512, 256.
    def __init__(self, in_dim=512, num_classes=NUM_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim * num_classes, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 2),
        )
    def forward(self, x):
        return self.net(x)


def multilinear_map(feat, pred):
    # KHONG detach feat: gradient doi khang phai qua GRL toi phi.
    return torch.bmm(feat.unsqueeze(2), pred.unsqueeze(1)).view(feat.size(0), -1)


def cdan_entropy_weight(probs):
    # w = 1 + exp(-H(p)); detach -> chi la he so.
    ent = -(probs * torch.log(probs + 1e-5)).sum(dim=1)
    return (1.0 + torch.exp(-ent)).detach()


# ── Mean Teacher ─────────────────────────────────────────────────────────────
EMA_DECAY     = 0.99
CONS_MAX      = float(NUM_CLASSES ** 2)   # = C^2 (notebook tao: 9 voi C=3)
CONS_START    = 100
CONS_RAMP_LEN = 50


def sigmoid_rampup(x, length):
    if length == 0:
        return 1.0
    x = float(np.clip(x, 0.0, length)) / length
    return float(np.exp(-5.0 * (1.0 - x) ** 2))


def consistency_weight(epoch, cons_max=CONS_MAX, start=CONS_START, ramp_len=CONS_RAMP_LEN):
    return 0.0 if epoch < start else cons_max * sigmoid_rampup(epoch - start, ramp_len)


def softmax_mse_loss(input_logits, target_logits):
    # Tong tren batch / num_classes; cho goi chia them batch size.
    return F.mse_loss(F.softmax(input_logits, 1), F.softmax(target_logits, 1),
                      reduction='sum') / input_logits.size(1)


def create_teacher(student_phi, student_G, device):
    phi_t = FeatureExtractor(pretrained=False).to(device)
    G_t   = Classifier(phi_t.out_dim, NUM_CLASSES).to(device)
    phi_t.load_state_dict(student_phi.state_dict())
    G_t.load_state_dict(student_G.state_dict())
    for p in list(phi_t.parameters()) + list(G_t.parameters()):
        p.requires_grad_(False)
    return phi_t, G_t


def update_ema(student_mods, teacher_mods, global_step, ema_decay=EMA_DECAY):
    # Chi .parameters(); BN buffer cua teacher cap nhat qua forward o .train().
    alpha = min(1 - 1 / (global_step + 1), ema_decay)
    with torch.no_grad():
        for sm, tm in zip(student_mods, teacher_mods):
            for ema_p, p in zip(tm.parameters(), sm.parameters()):
                ema_p.data.mul_(alpha).add_(p.data, alpha=1 - alpha)


# ── Danh gia: tong, theo lop, macro, confusion ──────────────────────────────
def evaluate(phi, G, loader, device):
    phi.eval(); G.eval()
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)   # [true, pred]
    with torch.no_grad():
        for imgs, lbls in loader:
            preds = G(phi(imgs.to(device))).argmax(1).cpu().numpy()
            for t, p in zip(lbls.numpy(), preds):
                cm[t, p] += 1
    per_class = {c: (100.0 * cm[i, i] / cm[i].sum() if cm[i].sum() else float('nan'))
                 for i, c in enumerate(CLASS_NAMES)}
    return {
        'acc': 100.0 * np.trace(cm) / cm.sum(),
        'macro_acc': float(np.nanmean(list(per_class.values()))),
        'per_class': per_class,
        'confusion': cm.tolist(),
    }

print(f'Models & helpers defined. NUM_CLASSES={NUM_CLASSES}, CONS_MAX={CONS_MAX}')

Models & helpers defined. NUM_CLASSES=2, CONS_MAX=4.0


## 6 — Một vòng lặp cho cả 5 nhánh

`train_arm(arm, seed)` là vòng lặp `train_cdane` của notebook táo, tổng quát hoá bằng
hai cờ `use_cdan` / `use_mt`. Giữ nguyên bốn điểm ghép cặp RNG:

1. Thứ tự khởi tạo `FeatureExtractor → Classifier → CDANDisc` ở **mọi** nhánh
   (nhánh không dùng D vẫn khởi tạo nó để tiêu thụ RNG giống nhau).
2. Teacher dựng trong `fork_rng` ở mọi nhánh.
3. `g.manual_seed(seed*1000 + epoch)` mỗi epoch.
4. View của target sinh trong `fork_rng`, seed theo `(DATA_SEED, epoch, lượt)`.

**Thay đổi duy nhất về cấu trúc:** một epoch = một lượt qua **source**
(~14 batch). Target chỉ có ~3 batch nên được lặp lại khi hết; mỗi lượt lặp gọi
`set_epoch(epoch*100 + lượt)` để hai view không lặp lại y hệt trong cùng epoch.

Chọn model: `val_loss` trên source val, từ epoch `MIN_SEL` — đo bằng **student** cho
nhánh không có MT, **teacher (EMA)** cho nhánh có MT (giống notebook táo).

`probe_epochs=N` chạy **đúng vòng lặp này** nhưng bỏ forward/backward, chỉ ghi chữ ký
batch source — đó là cách cell `rng_probe` kiểm chứng ghép cặp mà không cần code riêng.

In [20]:
ARMS = {
    #  arm        : (source, use_cdan, use_mt, ten hien thi)
    'src_raw'  : ('raw', False, False, 'Source-only (PV goc, no FBR)'),
    'src_fbr'  : ('fbr', False, False, 'Source-only w/ FBR'),
    'cdane'    : ('fbr', True,  False, 'CDAN+E w/ FBR'),
    'mt_only'  : ('fbr', False, True,  'MT only w/ FBR'),
    'cdane_mt' : ('fbr', True,  True,  'CDAN+E + MT w/ FBR'),
}
PAIRED_ARMS = ['src_fbr', 'cdane', 'mt_only', 'cdane_mt']   # cung source FBR -> phai cung dong RNG


def _target_batches(epoch):
    # Lap target loader cho du so batch cua source. Moi luot moi -> view moi.
    n_pass = 0
    while True:
        dataset_tgt_3v.set_epoch(epoch * 100 + n_pass)
        for batch in target_loader_3v:
            yield batch
        n_pass += 1


def save_ckpt(path, epoch, step, best_loss, best_state, phi, G, D, opt, sch, phi_t, G_t):
    torch.save({'epoch': epoch, 'step': step, 'best_loss': best_loss, 'best_state': best_state,
                'phi': phi.state_dict(), 'G': G.state_dict(), 'D': D.state_dict(),
                'optimizer': opt.state_dict(), 'scheduler': sch.state_dict(),
                # state_dict cua teacher gom ca BN buffer -> bat buoc luu de resume dung.
                'phi_t': phi_t.state_dict(), 'G_t': G_t.state_dict()}, path)


def load_ckpt(path, phi, G, D, opt, sch, phi_t, G_t):
    if not os.path.exists(path):
        return 1, 0, float('inf'), None
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    phi.load_state_dict(ck['phi']); G.load_state_dict(ck['G']); D.load_state_dict(ck['D'])
    opt.load_state_dict(ck['optimizer']); sch.load_state_dict(ck['scheduler'])
    phi_t.load_state_dict(ck['phi_t']); G_t.load_state_dict(ck['G_t'])
    return ck['epoch'] + 1, ck['step'], ck['best_loss'], ck['best_state']


def train_arm(arm, seed, max_epochs=MAX_EPOCHS, min_sel=MIN_SEL, ckpt_dir=CKPT_DIR,
              log_every=LOG_EVERY, probe_epochs=None, verbose=True):
    src_key, use_cdan, use_mt, _ = ARMS[arm]
    train_loader_src, val_loader_src = SOURCE[src_key]
    set_seed(seed)

    # (1) Thu tu khoi tao giong nhau o moi nhanh.
    phi = FeatureExtractor(pretrained=True).to(DEVICE)
    G   = Classifier(phi.out_dim, NUM_CLASSES).to(DEVICE)
    D   = CDANDisc(phi.out_dim, NUM_CLASSES).to(DEVICE)
    grl = GradRevLayer()
    # (2) Teacher trong fork_rng.
    with torch.random.fork_rng(devices=[]):
        phi_t, G_t = create_teacher(phi, G, DEVICE)

    params = list(phi.parameters()) + list(G.parameters()) + (list(D.parameters()) if use_cdan else [])
    opt = optim.AdamW(params, lr=LR, weight_decay=WD)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
    ce_mean, ce_persam = nn.CrossEntropyLoss(), nn.CrossEntropyLoss(reduction='none')

    n_epochs = probe_epochs or max_epochs
    ckpt_path = best_path = None
    start_epoch, step, best_loss, best_state = 1, 0, float('inf'), None
    if ckpt_dir is not None and probe_epochs is None:
        ckpt_path = os.path.join(ckpt_dir, f'{arm}_s{seed}_ckpt.pth')
        best_path = os.path.join(ckpt_dir, f'{arm}_s{seed}_best.pth')
        start_epoch, step, best_loss, best_state = load_ckpt(
            ckpt_path, phi, G, D, opt, sch, phi_t, G_t)
        if start_epoch > 1 and verbose:
            print(f'  [resume] {arm} s{seed}: tiep tuc tu epoch {start_epoch}')

    steps_per_epoch = len(train_loader_src)
    total_steps     = max_epochs * steps_per_epoch
    sel_phi, sel_G  = (phi_t, G_t) if use_mt else (phi, G)
    collapse_share  = 100 * (0.5 + 0.5 / NUM_CLASSES)
    sigs, lam       = [], 0.0

    for epoch in range(start_epoch, n_epochs + 1):
        g.manual_seed(seed * 1000 + epoch)                    # (3)
        tgt_iter = _target_batches(epoch)                     # (4) ben trong
        phi.train(); G.train(); D.train()
        phi_t.train(); G_t.train()                            # BN buffer cua teacher
        w_cons = consistency_weight(epoch) if use_mt else 0.0
        ep = {'cls': 0.0, 'cdan': 0.0, 'cons': 0.0}
        tgt_hist = torch.zeros(NUM_CLASSES, dtype=torch.long)

        for si, sl in train_loader_src:
            (t_plain, t_v1, t_v2), _ = next(tgt_iter)        # moi nhanh deu doc target
            if probe_epochs is not None:
                sigs.append((round(float(si.sum()), 4), int(sl.sum())))
                continue

            si, sl = si.to(DEVICE), sl.to(DEVICE)
            lam = get_lambda(step, total_steps); grl.set_lam(lam)
            step += 1
            opt.zero_grad()

            sf = phi(si)
            logits_s = G(sf)
            L_cls = ce_mean(logits_s, sl)
            total = L_cls

            if use_cdan:
                t_plain = t_plain.to(DEVICE)
                tf = phi(t_plain)
                logits_t = G(tf)
                sp, tp = torch.softmax(logits_s, 1), torch.softmax(logits_t, 1)
                src_mm = grl(multilinear_map(sf, sp.detach()))
                tgt_mm = grl(multilinear_map(tf, tp.detach()))
                sw = cdan_entropy_weight(sp); sw = sw / sw.sum()
                tw = cdan_entropy_weight(tp); tw = tw / tw.sum()
                src_d = (sw * ce_persam(D(src_mm), domain_lbl(si.size(0), True, DEVICE))).sum()
                tgt_d = (tw * ce_persam(D(tgt_mm), domain_lbl(t_plain.size(0), False, DEVICE))).sum()
                L_cdan = (src_d + tgt_d) / 2.0
                total = total + L_cdan
                ep['cdan'] += L_cdan.item()
                probe = logits_t.detach()

            if use_mt:
                t_v1, t_v2 = t_v1.to(DEVICE), t_v2.to(DEVICE)
                student_logits = G(phi(t_v1))
                with torch.no_grad():
                    teacher_logits = G_t(phi_t(t_v2))
                L_cons = softmax_mse_loss(student_logits, teacher_logits) / t_v1.size(0)
                total = total + w_cons * L_cons
                ep['cons'] += L_cons.item()
                probe = teacher_logits

            if use_cdan or use_mt:
                tgt_hist += torch.bincount(probe.argmax(1).cpu(), minlength=NUM_CLASSES)

            total.backward()
            opt.step()
            if use_mt:
                update_ema([phi, G], [phi_t, G_t], step)
            ep['cls'] += L_cls.item()

        if probe_epochs is not None:
            continue
        sch.step()

        # Validation tren source val (khong dung nhan target).
        sel_phi.eval(); sel_G.eval()
        vl, vc, vt = 0.0, 0, 0
        with torch.no_grad():
            for vi, vlab in val_loader_src:
                vi, vlab = vi.to(DEVICE), vlab.to(DEVICE)
                logits = sel_G(sel_phi(vi))
                vl += ce_mean(logits, vlab).item()
                vc += (logits.argmax(1) == vlab).sum().item()
                vt += vlab.size(0)
        val_loss, val_acc = vl / len(val_loader_src), vc / vt

        if epoch >= min_sel and val_loss < best_loss:
            best_loss  = val_loss
            best_state = {'phi': copy.deepcopy(sel_phi.state_dict()),
                          'G':   copy.deepcopy(sel_G.state_dict())}
            if best_path:
                torch.save(best_state, best_path)

        if verbose and (epoch % log_every == 0 or epoch == max_epochs):
            msg = (f'  [{arm} s{seed}] ep {epoch:03d}/{max_epochs} val_acc={val_acc*100:5.1f}% '
                   f'lam={lam:.3f} w={w_cons:.2f} | cls={ep["cls"]/steps_per_epoch:.3f} '
                   f'cdan={ep["cdan"]/steps_per_epoch:.3f} cons={ep["cons"]/steps_per_epoch:.4f}')
            if tgt_hist.sum() > 0:
                share = tgt_hist.float() / tgt_hist.sum() * 100
                msg += ' | tgt%=[' + ','.join(f'{s:.0f}' for s in share) + ']'
                if share.max().item() > collapse_share:
                    msg += f'\n     CANH BAO: target du doan don ve 1 lop ({share.max():.0f}%)'
            print(msg)

        if ckpt_path and (epoch % log_every == 0 or epoch == max_epochs):
            save_ckpt(ckpt_path, epoch, step, best_loss, best_state, phi, G, D, opt, sch, phi_t, G_t)

    if probe_epochs is not None:
        return sigs

    if best_state is None:
        print(f'  CANH BAO [{arm}]: chua toi cua so chon model, dung weight cuoi.')
        best_state = {'phi': copy.deepcopy(sel_phi.state_dict()), 'G': copy.deepcopy(sel_G.state_dict())}

    eval_phi = FeatureExtractor(pretrained=False).to(DEVICE)
    eval_G   = Classifier(eval_phi.out_dim, NUM_CLASSES).to(DEVICE)
    eval_phi.load_state_dict(best_state['phi']); eval_G.load_state_dict(best_state['G'])

    res = evaluate(eval_phi, eval_G, test_loader, DEVICE)
    res.update({'arm': arm, 'seed': seed, 'data_seed': DATA_SEED, 'task': TASK,
                'best_val_loss': best_loss, 'epochs_run': max_epochs,
                'acc_student_final': evaluate(phi, G, test_loader, DEVICE)['acc']})
    if use_mt and res['acc_student_final'] > res['acc'] + 3.0:
        print(f'  CANH BAO: student ({res["acc_student_final"]:.2f}%) hon han teacher '
              f'({res["acc"]:.2f}%) - nghi ngo BN buffer chua sync')
    return res

print('train_arm() defined. Arms:', list(ARMS))

train_arm() defined. Arms: ['src_raw', 'src_fbr', 'cdane', 'mt_only', 'cdane_mt']


## 7 — Kiểm chứng ghép cặp RNG (chạy trước, vài phút, không huấn luyện)

Bốn nhánh dùng source FBR phải thấy **cùng** chuỗi ảnh source (cùng thứ tự, cùng
augmentation). `[FAIL]` nghĩa là chênh lệch accuracy giữa các nhánh không còn quy được
về loss — đừng chạy 300 epoch.

In [21]:
_ref = train_arm(PAIRED_ARMS[0], SEEDS[0], probe_epochs=2, verbose=False)
print(f'So batch source doi chieu: {len(_ref)}')
for _arm in PAIRED_ARMS[1:]:
    _sig = train_arm(_arm, SEEDS[0], probe_epochs=2, verbose=False)
    if _sig == _ref:
        print(f'  [OK]   {PAIRED_ARMS[0]} == {_arm}')
    else:
        _i = next(i for i, (a, b) in enumerate(zip(_ref, _sig)) if a != b)
        print(f'  [FAIL] {PAIRED_ARMS[0]} vs {_arm}: lech tu batch {_i}: {_ref[_i]} vs {_sig[_i]}')
    assert _sig == _ref, f'{_arm} KHONG ghep cap voi {PAIRED_ARMS[0]} - dung chay tiep.'
del _ref, _sig

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 176MB/s]


So batch source doi chieu: 28
  [OK]   src_fbr == cdane
  [OK]   src_fbr == mt_only
  [OK]   src_fbr == cdane_mt


## 8 — Chạy tất cả nhánh × seed

Mỗi `(nhánh, seed)` ghi một file JSON trong `RESULTS_DIR`; đã có thì bỏ qua (xoá file
để chạy lại). Colab ngắt giữa chừng thì chạy lại cell này — checkpoint mỗi
`LOG_EVERY` epoch, tự resume.

Muốn chạy bớt nhánh, sửa `RUN_ARMS`. Thời gian tham khảo trên T4: nhánh `src_*` nhanh
nhất, `cdane_mt` chậm nhất (4 forward pass / iteration).

In [22]:
RUN_ARMS = ['cdane_mt']        # vd: ['src_raw', 'cdane', 'cdane_mt']


def result_path(arm, seed):
    return os.path.join(RESULTS_DIR, f'{arm}_s{seed}.json')


for seed in SEEDS:
    for arm in RUN_ARMS:
        path = result_path(arm, seed)
        if os.path.exists(path):
            print(f'[skip] {os.path.basename(path)} da co')
            continue
        print('=' * 64)
        print(f'ARM: {ARMS[arm][3]}  (seed={seed}, task={TASK})')
        print('=' * 64)
        res = train_arm(arm, seed)
        with open(path, 'w') as f:
            json.dump(res, f, indent=2)
        pc = ', '.join(f'{c}={v:.2f}%' for c, v in res['per_class'].items())
        print(f'-> test PlantDoc: acc={res["acc"]:.2f}%  macro={res["macro_acc"]:.2f}%  ({pc})')

ARM: CDAN+E + MT w/ FBR  (seed=42, task=potato)
  [cdane_mt s42] ep 010/300 val_acc= 94.7% lam=0.164 w=0.00 | cls=0.093 cdan=0.690 cons=0.0856 | tgt%=[47,53]
  [cdane_mt s42] ep 020/300 val_acc= 98.7% lam=0.320 w=0.00 | cls=0.069 cdan=0.689 cons=0.1010 | tgt%=[45,55]
  [cdane_mt s42] ep 030/300 val_acc= 98.7% lam=0.461 w=0.00 | cls=0.023 cdan=0.680 cons=0.0907 | tgt%=[41,59]
  [cdane_mt s42] ep 040/300 val_acc= 98.7% lam=0.582 w=0.00 | cls=0.006 cdan=0.618 cons=0.0592 | tgt%=[43,57]
  [cdane_mt s42] ep 050/300 val_acc= 98.7% lam=0.682 w=0.00 | cls=0.028 cdan=0.684 cons=0.0643 | tgt%=[46,54]
  [cdane_mt s42] ep 060/300 val_acc= 98.7% lam=0.761 w=0.00 | cls=0.011 cdan=0.734 cons=0.0788 | tgt%=[45,55]
  [cdane_mt s42] ep 070/300 val_acc= 98.0% lam=0.823 w=0.00 | cls=0.012 cdan=0.697 cons=0.0556 | tgt%=[43,57]
  [cdane_mt s42] ep 080/300 val_acc= 98.7% lam=0.870 w=0.00 | cls=0.008 cdan=0.686 cons=0.0948 | tgt%=[45,55]
  [cdane_mt s42] ep 090/300 val_acc= 98.7% lam=0.905 w=0.00 | cls=0.030 

## 9 — So sánh

Bảng 1: mean ± std qua các seed, mỗi nhánh. Bảng 2: chênh lệch **ghép cặp** (cùng
seed) giữa các nhánh — đây là con số nên dùng để kết luận, vì hai nhánh trong một
cặp chỉ khác nhau đúng một thành phần.

Dòng **Bài 5** là tham chiếu ngoài, *không* cùng điều kiện: attention-CNN (không phải
ResNet-18), ảnh 150×150, chia source 35/15. Đặt cạnh dòng `src_raw` (cùng setting:
train PV, test PlantDoc, không thích nghi) để thấy backbone/kiểu ảnh khác nhau đến đâu;
đặt cạnh các nhánh UDA để thấy pipeline của bạn cải thiện so với mức không thích nghi
đã công bố.

In [ ]:
_rows = {}
for arm in ARMS:
    rs = [json.load(open(result_path(arm, s))) for s in SEEDS if os.path.exists(result_path(arm, s))]
    if rs:
        _rows[arm] = rs

if not _rows:
    print('Chua co ket qua nao. Chay cell muc 8 truoc.')
else:
    def _ms(vals):
        return f'{np.mean(vals):6.2f}' + (f' ± {np.std(vals, ddof=1):5.2f}' if len(vals) > 1 else '        ')

    cols = ['acc', 'macro_acc'] + CLASS_NAMES
    w = 18
    print('=' * (36 + w * len(cols)))
    print(f'{"Method":<30}{"n":>4}  ' + ''.join(f'{c:>{w}}' for c in cols))
    print('-' * (36 + w * len(cols)))
    for arm, rs in _rows.items():
        vals = [_ms([r['acc'] for r in rs]), _ms([r['macro_acc'] for r in rs])]
        vals += [_ms([r['per_class'][c] for r in rs]) for c in CLASS_NAMES]
        print(f'{ARMS[arm][3]:<30}{len(rs):>4}  ' + ''.join(f'{v:>{w}}' for v in vals))
    ref = CFG['paper_ref']
    if ref:
        macro = np.mean(list(ref.values()))
        vals = ['—', f'{macro:6.2f}*'] + [f'{ref[c]:6.2f}' for c in CLASS_NAMES]
        print(f'{"Bai 5 (Table 21)":<30}{"":>4}  ' + ''.join(f'{v:>{w}}' for v in vals))
    print('=' * (36 + w * len(cols)))
    if ref:
        print('* macro cua bai 5 tu tinh = trung binh accuracy theo lop; bai bao khong bao cao.')
        print('  Bai 5 khong bao cao accuracy tong (phu thuoc so anh moi lop cua ho).')

    # ── Chenh lech ghep cap (cung seed) ─────────────────────────────────────
    PAIRS = [('src_fbr', 'src_raw', 'FBR'),
             ('cdane', 'src_fbr', 'CDAN+E (tren FBR)'),
             ('mt_only', 'src_fbr', 'MT (tren FBR)'),
             ('cdane_mt', 'cdane', 'them MT vao CDAN+E'),
             ('cdane_mt', 'mt_only', 'them CDAN+E vao MT')]
    print('\nCHENH LECH GHEP CAP (cung seed), don vi: diem %')
    print('-' * 72)
    for a, b, label in PAIRS:
        if a in _rows and b in _rows:
            ra = {r['seed']: r for r in _rows[a]}
            rb = {r['seed']: r for r in _rows[b]}
            common = sorted(set(ra) & set(rb))
            if not common:
                continue
            d_acc   = [ra[s]['acc'] - rb[s]['acc'] for s in common]
            d_macro = [ra[s]['macro_acc'] - rb[s]['macro_acc'] for s in common]
            print(f'  {label:<22} {a:>9} - {b:<8}: acc {_ms(d_acc)} | macro {_ms(d_macro)}  (n={len(common)})')

    # ── Confusion matrix cua seed dau tien ──────────────────────────────────
    print(f'\nCONFUSION MATRIX (seed {SEEDS[0]}, hang = nhan that, cot = du doan; thu tu {CLASS_NAMES})')
    for arm, rs in _rows.items():
        r0 = next((r for r in rs if r['seed'] == SEEDS[0]), rs[0])
        print(f'  {ARMS[arm][3]:<26} {r0["confusion"]}')

### Cách phát biểu kết quả trong luận văn

1. **So với bài 5 là so lỏng.** Cùng dataset, cùng hướng lab → field, cùng phần test
   50%, nhưng khác backbone, kích thước ảnh và cách chia source. Phát biểu: *"Trên cùng
   cặp PlantVillage → PlantDoc (khoai tây), mô hình không thích nghi của Mahapatra et al.
   đạt X/Y theo lớp; pipeline của chúng tôi đạt …"*, không phát biểu *"vượt bài 5"*.
2. **Kết luận về từng thành phần dùng bảng ghép cặp**, không dùng bảng mean ± std:
   các nhánh trong một cặp có chung khởi tạo, thứ tự dữ liệu và augmentation source.
3. **Test chỉ ~111 ảnh**: mỗi ảnh ≈ 0.9 điểm. Với `n=1` seed, chênh lệch dưới vài điểm
   không kết luận được — chạy `SEEDS = [42, 43, 44, 45, 46]`.
4. Bài 5 cho thấy mô hình không thích nghi **dồn gần hết dự đoán về early blight**
   (94.87% vs 10.48%). Vì vậy `macro_acc` và accuracy theo lớp quan trọng hơn accuracy
   tổng ở đây; theo dõi cả dòng `tgt%` trong log để thấy nhánh nào thoát được việc dồn lớp.